# ABC （Abstract Base Class，简称 ABC）

- 在 Python 中，from abc import ABC, abstractmethod 是用来实现抽象基类（Abstract Base Class，简称 ABC）的标准方式。
简单来说，抽象基类就像是一份“强制执行的合同”或“建筑蓝图”。
- <font color='red'>它定义了一组必须遵守的规范，但本身不能被直接使用。任何继承了这个基类的“子类”，都必须实现基类中规定的所有抽象方法，否则程序在运行时就会直接报错。</font>

## 💡 为什么要使用 ABC？
<font color='red'>如果不使用 ABC，很多开发者会用 NotImplementedError 来模拟抽象约束，但这有一个致命缺陷：错误发现得太晚。</font>
- 不使用 ABC 的情况：子类忘记实现某个方法，代码在实例化时不会报错，只有当程序运行到调用那个方法时才会抛出异常。这在大型项目中很容易埋下隐性 Bug。
- 使用 ABC 的情况：如果子类没有实现所有的抽象方法，Python 会在实例化阶段（创建对象时）直接抛出 TypeError，强制你在写代码的早期就发现问题。
## 💻 基础代码示例
我们以一个经典的“图形计算”为例，定义一个图形基类，强制要求所有具体的图形（如圆形、矩形）都必须具备“计算面积”的能力：

In [ ]:
fron abc import ABC,abstractmethod

# 1. 定义抽象类，必须继承ABC
clss Shape(ABC):
    '''形状抽象基类'''
    
    # 2. 使用@abstractmethod装饰器标记抽象方法
    @abstractmethod
    def area(self):
        '''所有的子类必须实现这个方法'''
        pass
    # 抽象基类中也可以包含有具体实现的普通方法
    def describe(self):
        return f"这是一个{self.__class__.__name__}，面积为 {self.area()}"
    
# 3. 创建具体的子类
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    # 必须实现 area 方法，否则 Circle 也无法被实例化
    def area(self):
        return 3.14 * self.radius * self.radius

class Square(Shape):
    def __init__(self, side):
        self.side = side
    
    def area(self):
        return self.side * self.side

# --- 测试 ---

# ✅ 正常实例化，因为 Circle 和 Square 都实现了 area 方法
circle = Circle(5)
square = Square(4)
print(circle.describe())  # 输出: 这是一个Circle，面积为 78.5
print(square.describe())  # 输出: 这是一个Square，面积为 16

# ❌ 尝试实例化一个没有实现 area 的子类
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    # 忘记实现 area 方法了！

# 程序会在这里直接报错，阻止有缺陷的对象被创建
# 报错: TypeError: Can't instantiate abstract class Triangle with abstract method area
# tri = Triangle(3, 4) 

## 进阶用法：抽象属性、类方法与静态方法
- @abstractmethod 不仅可以装饰普通的实例方法，还可以装饰 @property、@classmethod 和 @staticmethod，用来强制子类实现特定的属性或工厂方法 

In [ ]:
from abc import ABC, abstractmethod

class DataProcessor(ABC):
    @property
    @abstractmethod
    def version(self):
        """强制子类提供一个只读属性 version"""
        pass

    @classmethod
    @abstractmethod
    def from_json(cls, json_data):
        """强制子类实现一个从 JSON 创建对象的类方法"""
        pass

class CSVProcessor(DataProcessor):
    def __init__(self, version):
        self._version = version

    # 实现抽象属性
    @property
    def version(self):
        return self._version

    # 实现抽象类方法
    @classmethod
    def from_json(cls, json_data):
        return cls(version=json_data.get("version", "1.0"))

# 正常使用
processor = CSVProcessor.from_json({"version": "2.0"})
print(processor.version) # 输出: 2.0

## 📌 总结
使用 ABC 和 @abstractmethod 是 Python 面向对象编程中一种非常优秀的设计模式。它的核心意义在于：
- 强制契约：把“你应该实现这个方法”的口头约定，变成了“你必须实现这个方法”的代码级强制约束。
- 提前拦截：在对象创建阶段就拦截错误，而不是等到运行时才崩溃。
- 代码自文档化：看到继承自 ABC 的类，其他开发者立刻就能明白这是一个接口模板，并清楚知道子类需要具备哪些核心功能。